# RepoCoder Studio — RAG Transformation Continuation v1.3

This notebook corrects the verified v1.2 boundary without overwriting v1.2. It starts from the saved **v1.2 LoRA adapter**, performs a smaller continuation run, and saves a separate **v1.3-transform** adapter.

The continuation data contains three balanced signals:

1. replay of validated v1.2 examples to reduce catastrophic forgetting;
2. repository-aware wrapper/composition targets that call rather than copy evidence;
3. repository-aware in-place extensions that preserve existing logic while applying a new requested rule.

Run the cells in order. Step 1 restarts Colab once. Steps 2–5 are CPU preparation, Step 6 is the only training step, Step 7 verifies both previously observed failure modes, and Step 9 launches the corrected Gradio UI.


## Step 1 — Install dependencies and restart

Run once. Colab restarts automatically; continue with Step 2 after reconnecting.


In [ ]:
# ============================================================
# Step 1a — Install dependencies, then restart
# ============================================================

import sys
import shutil
import subprocess

if shutil.which("javac") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"])

packages = [
    "numpy==1.26.4",
    "huggingface_hub==0.25.2",
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "peft==0.12.0",
    "trl==0.10.1",
    "bitsandbytes>=0.43,<0.46",
    "sentencepiece",
    "protobuf>=3.20.2,<6",
    "sentence-transformers>=3.0,<4",
    "tree-sitter==0.22.3",
    "tree-sitter-python==0.21.0",
    "tree-sitter-java==0.21.0",
    "faiss-cpu>=1.8,<2",
    "networkx",
    "matplotlib",
    "pandas",
    "fastapi==0.112.2",
    "starlette==0.38.6",
    "uvicorn==0.30.6",
    "gradio-client==1.3.0",
    "gradio==4.44.1",
]

print("Installing RepoCoderStudio runtime dependencies...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir", *packages]
)

print("Install complete. Restarting the kernel to load the new numpy build...")
print("After the restart, run the next cell (do not rerun this one).")

import os
os.kill(os.getpid(), 9)


Installing RepoCoderStudio runtime dependencies...


## Step 2 — Mount Drive and verify all prerequisites

This cell selects v1.3 as the new output identity while requiring the existing v1.2 adapter as its immutable parent. Nothing in the v1.2 directory is changed.


In [1]:
# ============================================================
# Step 2 — Mount Drive, configure v1.3, verify prerequisites
# ============================================================

import os
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/RepoCoderStudio").resolve()
assert PROJECT_ROOT.is_dir(), f"Project folder not found: {PROJECT_ROOT}"

# v1.3 is a new child adapter. The saved v1.2 parent remains untouched.
V12_ADAPTER_NAME = "RepoCoderStudio_RAGAware_LoRA_v1_2"
V13_ADAPTER_NAME = "RepoCoderStudio_RAGAware_LoRA_v1_3_transform"

os.environ["REPOCODER_ADAPTER_NAME"] = V13_ADAPTER_NAME
os.environ["REPOCODER_PROMPT_VERSION"] = "rag_prompt_contract_v1.3_transform"
os.environ["REPOCODER_TRAINING_MANIFEST_VERSION"] = "training_manifest_rag_v1.3_transform"
os.environ["REPOCODER_TRAINING_DATASET_FILENAME"] = "task_dataset_rag_transform_v1_3.jsonl"
os.environ["REPOCODER_LEARNING_RATE"] = "2e-5"
os.environ["REPOCODER_MAX_CONTEXT_CHARS"] = "2000"
os.environ["REPOCODER_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["REPOCODER_RUN_MODE"] = "demo"
os.environ["REPOCODER_ENABLE_RAG"] = "true"
os.environ["REPOCODER_ALLOW_MOCK_EMBEDDINGS"] = "false"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "false"

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_source_files = [
    PROJECT_ROOT / "src" / "task_builder_rag_transform.py",
    PROJECT_ROOT / "src" / "continuation_trainer.py",
    PROJECT_ROOT / "src" / "response_safe_training_rag.py",
    PROJECT_ROOT / "src" / "browser_code_runner.py",
    PROJECT_ROOT / "src" / "gradio_showcase.py",
]
missing_source = [str(path) for path in required_source_files if not path.is_file()]
assert not missing_source, "Copy these corrected source files to Drive:\n" + "\n".join(missing_source)

import torch
import faiss
import gradio as gr

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

from src.config import CONFIG
from src.storage import ProjectStorageManager
from src.repository_catalog import repository_index_dirs

storage = ProjectStorageManager(CONFIG)
V12_ADAPTER_DIR = PROJECT_ROOT / CONFIG.storage.adapters_dir / V12_ADAPTER_NAME
V12_TASK_DATASET = PROJECT_ROOT / "outputs/task_datasets/task_dataset_rag_grounded.jsonl"

required_artifacts = {
    "v1.2 parent adapter": V12_ADAPTER_DIR / "adapter_model.safetensors",
    "v1.2 adapter manifest": V12_ADAPTER_DIR / "trained_model_manifest.json",
    "v1.2 task dataset": V12_TASK_DATASET,
    "approved corpus": storage.path(storage.approved_corpus_path()),
    "Stage 5 corpus index": storage.corpus_index_dir() / "corpus_index_manifest.json",
}
_, ledgerflow_embedding_dir = repository_index_dirs("ledgerflow", config=CONFIG)
required_artifacts["Stage 4 LedgerFlow index"] = ledgerflow_embedding_dir / "index_manifest.json"
missing = {name: str(path) for name, path in required_artifacts.items() if not path.is_file()}
assert not missing, "Missing prerequisites:\n" + "\n".join(f"{k}: {v}" for k, v in missing.items())

assert CONFIG.training.final_adapter_name == V13_ADAPTER_NAME
assert CONFIG.training.learning_rate == 2e-5
print("Parent adapter (read-only):", V12_ADAPTER_DIR)
print("New adapter output       :", CONFIG.training.final_adapter_name)
print("All prerequisites        : PASS")


Mounted at /content/drive
GPU: Tesla T4
Parent adapter (read-only): /content/drive/MyDrive/RepoCoderStudio/outputs/adapters/RepoCoderStudio_RAGAware_LoRA_v1_2
New adapter output       : RepoCoderStudio_RAGAware_LoRA_v1_3_transform
All prerequisites        : PASS


## Step 3 — Load the validated corpus and v1.2 task records

The notebook reuses the already-approved data; it does not rebuild the corpus or retrain v1.2.


In [2]:
# ============================================================
# Step 3 — Load approved corpus and reusable v1.2 task records
# ============================================================

import json
from src.schemas import ApprovedRow, TaskExample

approved_rows = [ApprovedRow(**row) for row in storage.load_jsonl(storage.approved_corpus_path())]
v12_task_examples = [TaskExample(**row) for row in storage.load_jsonl(V12_TASK_DATASET)]

v12_manifest = json.loads(
    (V12_ADAPTER_DIR / "trained_model_manifest.json").read_text(encoding="utf-8")
)
assert v12_manifest.get("adapter_name") == V12_ADAPTER_NAME, v12_manifest
assert any(ex.metadata.get("rag_augmented") for ex in v12_task_examples if ex.split == "train")

print("Approved rows       :", len(approved_rows))
print("v1.2 task examples  :", len(v12_task_examples))
print("v1.2 parent manifest: PASS")


07:14:35 | INFO     | RepoCoderStudio.Main | Loaded JSONL: /content/drive/MyDrive/RepoCoderStudio/outputs/approved_corpus/approved_corpus.jsonl (537 rows)
07:14:38 | INFO     | RepoCoderStudio.Main | Loaded JSONL: /content/drive/MyDrive/RepoCoderStudio/outputs/task_datasets/task_dataset_rag_grounded.jsonl (3789 rows)
Approved rows       : 537
v1.2 task examples  : 3789
v1.2 parent manifest: PASS


## Step 4 — Build balanced replay, composition and extension training rows

Replay preserves existing skills. The two transformation groups directly address the independently reproduced copy-collapse failures. LedgerFlow remains unseen during training.


In [4]:
# ============================================================
# Step 4 — Build balanced continuation dataset
# ============================================================

import importlib
import random
from collections import Counter, defaultdict

import src.task_builder_rag_transform as transform_module

# Reload the replaced source file without restarting Colab.
transform_module = importlib.reload(transform_module)
build_transform_augmented_examples = (
    transform_module.build_transform_augmented_examples
)

from src.schemas import dataclass_to_dict

# The approved corpus contains 159 functions that satisfy all strict
# transformation-safety requirements. The same validated function may
# legitimately provide one composition example and one extension example.
#
# This produces:
#   150 composition rows
#   150 in-place-extension rows
#   300 transformation rows in total
transform_examples = build_transform_augmented_examples(
    approved_rows=approved_rows,
    reference_examples=v12_task_examples,
    compose_count=150,
    extension_count=150,
    seed=31,
    context_char_budget=2000,
)

# ------------------------------------------------------------
# Deterministic task-balanced replay
# ------------------------------------------------------------
# Keep 60 examples from every original task to preserve the skills
# already learned by v1.2 and reduce catastrophic forgetting.

rng = random.Random(43)
by_task = defaultdict(list)

for example in v12_task_examples:
    if example.split == "train":
        by_task[example.task_id].append(example)

replay_examples = []

for task_id in sorted(by_task):
    candidates = list(by_task[task_id])
    rng.shuffle(candidates)

    selected = candidates[:60]
    replay_examples.extend(selected)

# Preserve the original validation and test records.
validation_and_test = [
    example
    for example in v12_task_examples
    if example.split in {"validation", "test"}
]

task_examples = (
    replay_examples
    + transform_examples
    + validation_and_test
)

# ------------------------------------------------------------
# Transformation-data integrity checks
# ------------------------------------------------------------

transform_counts = Counter(
    example.metadata.get("transform_type")
    for example in transform_examples
)

assert transform_counts == {
    "composition": 150,
    "inplace_extension": 150,
}, transform_counts

assert all(
    example.split == "train"
    for example in transform_examples
)

assert all(
    example.task_id == "T1"
    for example in transform_examples
)

assert all(
    "ledgerflow"
    not in example.metadata["retrieved_context"].lower()
    for example in transform_examples
), "LedgerFlow leaked into transformation training."

assert all(
    example.output_text.strip()
    not in example.metadata["retrieved_context"]
    for example in transform_examples
), "A transformation target collapsed into evidence reproduction."

assert all(
    len(example.metadata["retrieved_context"]) <= 2000
    for example in transform_examples
)

# ------------------------------------------------------------
# Save the complete v1.3 continuation dataset
# ------------------------------------------------------------

dataset_relative_path = (
    "outputs/task_datasets/"
    "task_dataset_rag_transform_v1_3.jsonl"
)

storage.save_jsonl(
    [
        dataclass_to_dict(example)
        for example in task_examples
    ],
    dataset_relative_path,
)

print()
print("=" * 70)
print("v1.3 Transformation Dataset")
print("=" * 70)
print("Eligible source functions :", 159)
print("Replay rows              :", len(replay_examples))
print("Composition rows         :", transform_counts["composition"])
print("Extension rows           :", transform_counts["inplace_extension"])
print("Total transform rows     :", len(transform_examples))
print("Validation/test rows     :", len(validation_and_test))
print("All task examples        :", len(task_examples))
print("Dataset path             :", dataset_relative_path)
print("Training preflight       : PASS")
print("=" * 70)

07:15:18 | INFO     | RepoCoderStudio.Main | Saved JSONL: /content/drive/MyDrive/RepoCoderStudio/outputs/task_datasets/task_dataset_rag_transform_v1_3.jsonl (1452 rows)

v1.3 Transformation Dataset
Eligible source functions : 159
Replay rows              : 360
Composition rows         : 150
Extension rows           : 150
Total transform rows     : 300
Validation/test rows     : 792
All task examples        : 1452
Dataset path             : outputs/task_datasets/task_dataset_rag_transform_v1_3.jsonl
Training preflight       : PASS


## Step 5 — Tokenize and apply response-safe budgeting

Only input/context may be truncated. Completion labels and response headers must remain intact.


In [8]:
# ============================================================
# Step 5 — Tokenize and apply response-safe budgeting
# ============================================================

from collections import Counter

from src.tokenizer_builder import TokenizerDatasetBuilder
from src.response_safe_training_rag import (
    apply_response_safe_budgeting,
)

tokenizer_builder = TokenizerDatasetBuilder(CONFIG)

train_dataset, validation_dataset, test_dataset = (
    tokenizer_builder.build(task_examples)
)

train_rows_before_budgeting = len(train_dataset)

print()
print("Train rows before budgeting :", train_rows_before_budgeting)
print("Expected replay rows        :", len(replay_examples))
print("Expected composition rows   :", transform_counts["composition"])
print(
    "Expected extension rows     :",
    transform_counts["inplace_extension"],
)

train_dataset, budget_report = apply_response_safe_budgeting(
    train_dataset,
    task_examples,
    config=CONFIG,
    report_path=(
        "outputs/reports/"
        "training_sequence_budget_report_rag_transform_v1_3.json"
    ),
)

train_rows_after_budgeting = len(train_dataset)
excluded_rows = (
    train_rows_before_budgeting
    - train_rows_after_budgeting
)

# ------------------------------------------------------------
# Completion-safety checks
# ------------------------------------------------------------

assert (
    budget_report["response_header_preflight_failures"] == 0
), budget_report

assert (
    budget_report["maximum_retained_sequence_tokens"]
    <= CONFIG.models.max_seq_length
), budget_report

assert (
    budget_report["retained_rows"]
    == train_rows_after_budgeting
)

assert (
    budget_report["excluded_rows"]
    == excluded_rows
)

# A complete response that cannot fit must be excluded.
# Require at least 90% overall retention.
assert budget_report["retention_rate"] >= 0.90, (
    "Too many training rows were excluded.",
    budget_report,
)

# ------------------------------------------------------------
# Count retained transformation rows correctly
# ------------------------------------------------------------
#
# Do not match using prompt_hash. Input truncation rebuilds the
# prompt and therefore intentionally changes its hash.
#
# Transformation rows have stable corpus-id suffixes:
#   _compose
#   _extend

assert "corpus_id" in train_dataset.column_names

retained_corpus_ids = [
    str(value)
    for value in train_dataset["corpus_id"]
]

retained_types = Counter(
    {
        "composition": sum(
            corpus_id.endswith("_compose")
            for corpus_id in retained_corpus_ids
        ),
        "inplace_extension": sum(
            corpus_id.endswith("_extend")
            for corpus_id in retained_corpus_ids
        ),
    }
)

# Retain enough examples from both correction families.
assert retained_types["composition"] >= 125, retained_types
assert retained_types["inplace_extension"] >= 125, retained_types

print()
print("=" * 70)
print("Response-Safe Training Dataset")
print("=" * 70)
print("Train rows before budgeting :", train_rows_before_budgeting)
print("Train rows after budgeting  :", train_rows_after_budgeting)
print("Rows safely excluded        :", excluded_rows)
print(
    "Retention rate             :",
    f"{budget_report['retention_rate']:.2%}",
)
print(
    "Composition rows retained  :",
    retained_types["composition"],
)
print(
    "Extension rows retained    :",
    retained_types["inplace_extension"],
)
print(
    "Maximum sequence tokens    :",
    budget_report["maximum_retained_sequence_tokens"],
)
print(
    "Response-header failures   :",
    budget_report["response_header_preflight_failures"],
)
print("Completion-label preflight  : PASS")
print("=" * 70)


Tokenizer Dataset Builder  [v2.6]

HF Dataset Summary  [v2.6]
Train Rows                  : 660
Validation Rows             : 366
Test Rows                   : 426
Train Task Counts           : {'T1': 360, 'T2': 60, 'T3': 60, 'T4': 60, 'T5': 60, 'T6': 60}
Validation Task Counts      : {'T1': 61, 'T2': 61, 'T3': 61, 'T4': 61, 'T5': 61, 'T6': 61}
Test Task Counts            : {'T1': 71, 'T2': 71, 'T3': 71, 'T4': 71, 'T5': 71, 'T6': 71}
Curriculum                  : round_robin_by_task

Train rows before budgeting : 660
Expected replay rows        : 360
Expected composition rows   : 150
Expected extension rows     : 150


Applying response-safe token budgeting (RAG-aware):   0%|          | 0/660 [00:00<?, ? examples/s]

Keeping completion-safe training rows:   0%|          | 0/660 [00:00<?, ? examples/s]


Response-Safe Training Dataset
Train rows before budgeting : 660
Train rows after budgeting  : 637
Rows safely excluded        : 23
Retention rate             : 96.52%
Composition rows retained  : 150
Extension rows retained    : 144
Maximum sequence tokens    : 1021
Response-header failures   : 0
Completion-label preflight  : PASS


## Step 6 — Continue v1.2 into a separate v1.3 adapter

This is the only GPU-heavy step. It attaches the saved v1.2 LoRA weights as trainable parameters, runs the smaller continuation dataset, and writes a new adapter directory. v1.2 is never overwritten.


In [9]:
# ============================================================
# Step 6 — Train/reuse the separate v1.3 continuation adapter
# ============================================================

import gc
import hashlib
import json

from src.artifact_manifest import file_sha256
from src.continuation_trainer import AdapterContinuationTrainer
from src.response_safe_training_rag import RESPONSE_SAFE_BUDGETING_RAG_VERSION

adapter_dir = PROJECT_ROOT / CONFIG.storage.adapters_dir / V13_ADAPTER_NAME
manifest_path = adapter_dir / "trained_model_manifest.json"
dataset_path = storage.path("outputs/task_datasets/task_dataset_rag_transform_v1_3.jsonl")
parent_manifest_sha = file_sha256(V12_ADAPTER_DIR / "trained_model_manifest.json")
budgeted_sha = hashlib.sha256(
    "\n".join(str(value) for value in train_dataset["prompt_hash"]).encode("utf-8")
).hexdigest()

expected_manifest = {
    "adapter_name": V13_ADAPTER_NAME,
    "parent_adapter_name": V12_ADAPTER_NAME,
    "parent_adapter_manifest_sha256": parent_manifest_sha,
    "base_model": CONFIG.models.student_model_name,
    "prompt_version": CONFIG.experiment.prompt_version,
    "training_manifest_version": CONFIG.experiment.training_manifest_version,
    "task_dataset_filename": CONFIG.training.task_dataset_filename,
    "task_dataset_sha256": file_sha256(dataset_path),
    "response_safe_budgeting_version": RESPONSE_SAFE_BUDGETING_RAG_VERSION,
    "budgeted_prompt_sha256": budgeted_sha,
    "train_rows": len(train_dataset),
    "training_strategy": "continue_v1.2_with_balanced_replay_and_grounded_transformations",
}

weights_exist = (adapter_dir / "adapter_model.safetensors").is_file()
saved_manifest = None
if manifest_path.is_file():
    try:
        saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        pass

if weights_exist and saved_manifest != expected_manifest:
    raise RuntimeError(
        f"An incompatible/interrupted v1.3 adapter exists at {adapter_dir}. "
        "Delete only that v1.3_transform folder and rerun this cell. Do not delete v1.2."
    )

if weights_exist and saved_manifest == expected_manifest:
    print("Reusing validated v1.3 adapter:", adapter_dir)
else:
    trainer_wrapper = AdapterContinuationTrainer(V12_ADAPTER_DIR, CONFIG)
    trainer_wrapper.load_model_and_tokenizer()
    trainer_wrapper.train(train_dataset, validation_dataset)
    adapter_dir.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(expected_manifest, indent=2), encoding="utf-8")
    print("Saved v1.3 adapter:", adapter_dir)

    globals().pop("trainer_wrapper", None)
    gc.collect()
    torch.cuda.empty_cache()

assert (V12_ADAPTER_DIR / "adapter_model.safetensors").is_file()
assert (adapter_dir / "adapter_model.safetensors").is_file()
print("v1.2 preserved and v1.3 available: PASS")



Loading Student Model


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

07:25:41 | INFO     | RepoCoderStudio.Main | Loaded model: Qwen/Qwen2.5-Coder-0.5B-Instruct

LoRA Training  [v2.7]
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


...

07:25:50 | INFO     | RepoCoderStudio.Main | No checkpoint found. Training will start fresh.


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,0.769600
20,0.611900
30,0.516100
40,0.545800
50,0.465800
60,0.515200
70,0.513500


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



Training Summary  [v2.7]
Train Rows                  : 637
Validation Rows             : 366
Final Adapter               : /content/drive/MyDrive/RepoCoderStudio/outputs/adapters/RepoCoderStudio_RAGAware_LoRA_v1_3_transform
Training History            : /content/drive/MyDrive/RepoCoderStudio/outputs/reports/training_history.csv
Training Summary            : /content/drive/MyDrive/RepoCoderStudio/outputs/reports/training_summary.json
Eval During Training        : Disabled
Loss Masking                : Completion tokens only
Saved v1.3 adapter: /content/drive/MyDrive/RepoCoderStudio/outputs/adapters/RepoCoderStudio_RAGAware_LoRA_v1_3_transform
v1.2 preserved and v1.3 available: PASS


## Step 7 — Restore v1.3 and verify reproduction, composition and extension

Success requires both previously failing transformation modes—not merely syntactic validity. The assertions stop the notebook if v1.3 still collapses to evidence copying.


In [13]:
# ============================================================
# Step 7 — Load v1.3 runtime and run verification gates
# ============================================================

from src.generation_engine import GenerationEngine
from src.generation_validation import GenerationOutputValidator
from src.registry import TaskRegistry
from src.repo_explorer import RepositoryExplorer
from src.repository_catalog import (
    repository_index_dirs,
    repository_path,
)
from src.corpus_retriever import CorpusIndex
from src.retrieval_engine import RetrievalEngine


# ------------------------------------------------------------
# Load baseline and v1.3 models
# ------------------------------------------------------------

engine = GenerationEngine(CONFIG)

print("Loading baseline model...")
baseline_model, baseline_tokenizer = engine.load_model(
    "baseline"
)

print("Loading v1.3 fine-tuned adapter...")
finetuned_model, finetuned_tokenizer = engine.load_model(
    "finetuned"
)


# ------------------------------------------------------------
# Restore the saved LedgerFlow repository and corpus indexes
# ------------------------------------------------------------

repo = repository_path(
    "ledgerflow",
    config=CONFIG,
)

parsed_dir, embedding_dir = repository_index_dirs(
    "ledgerflow",
    config=CONFIG,
)

explorer = RepositoryExplorer(
    repo_path=str(repo),
    output_dir=str(parsed_dir),
    embedding_dir=str(embedding_dir),
    model_name=CONFIG.retrieval.embedding_model,
    rebuild=False,
    config=CONFIG,
)

corpus_index = CorpusIndex(
    config=CONFIG,
    embedder=explorer.embedder,
)

corpus_rows = corpus_index.build(
    rebuild=False,
)

assert corpus_rows > 0, (
    "The saved Stage 5 corpus index contains no rows."
)

ledgerflow_retrieval = RetrievalEngine(
    explorer,
    CONFIG,
    corpus_index=corpus_index,
)

output_validator = GenerationOutputValidator(CONFIG)
task_registry = TaskRegistry()


# ------------------------------------------------------------
# Shared fine-tuned + repository-RAG generation function
# ------------------------------------------------------------

def generate_with_rag(query):
    task_id = "T1"

    outcome = ledgerflow_retrieval.resolve(
        query,
        task_id=task_id,
        top_k=1,
        sources=("repo",),
    )

    assert outcome.used, (
        f"Repository evidence was not accepted: "
        f"{outcome.decision}"
    )

    assert outcome.sources, (
        "No repository source was returned."
    )

    instruction = engine.prompt_builder.build_instruction(
        task_id
    )

    raw_output = engine.generate(
        finetuned_model,
        finetuned_tokenizer,
        instruction,
        query,
        task_id=task_id,
        retrieved_context=outcome.context,
    )

    checked = output_validator.validate(
        task_id,
        raw_output,
    )

    normalized_output = checked.get(
        "normalized_output",
        raw_output,
    )

    return (
        normalized_output,
        checked,
        outcome,
        raw_output,
    )


# ------------------------------------------------------------
# Policy-checking helpers
# ------------------------------------------------------------

def normalized_numeric_source(code):
    """
    Python treats numeric literals such as 250000 and 250_000
    identically. Remove cosmetic separators before comparison.
    """
    return str(code).replace("_", "")


def contains_original_transfer_policy(code):
    normalized = normalized_numeric_source(code)

    required_numbers = (
        "250000",
        "100000",
        "40",
        "25",
        "20",
        "30",
        "15",
        "100",
    )

    required_countries = (
        "IR",
        "KP",
        "SY",
    )

    return (
        "def transfer_risk_score" in code
        and "high_risk_countries" in code
        and all(
            number in normalized
            for number in required_numbers
        )
        and all(
            country in code
            for country in required_countries
        )
        and "min(" in code
    )


verification_results = {}


# ============================================================
# Test 1 — Grounded reproduction regression check
# ============================================================

print()
print("=" * 70)
print("TEST 1 — GROUNDED POLICY REPRODUCTION")
print("=" * 70)

reproduction_query = (
    "Implement transfer_risk_score("
    "amount, customer_tenure_days, "
    "destination_country, trusted_device"
    ") using this repository's exact transfer policy. "
    "Preserve its thresholds, weights, country set and "
    "100-point cap. Return Python code only."
)

(
    reproduction_code,
    reproduction_checked,
    reproduction_outcome,
    reproduction_raw,
) = generate_with_rag(reproduction_query)

reproduction_ok = (
    bool(reproduction_checked.get("valid"))
    and contains_original_transfer_policy(
        reproduction_code
    )
)

verification_results["grounded_reproduction"] = {
    "success": reproduction_ok,
    "status": reproduction_checked.get("status"),
    "reason": reproduction_checked.get("reason"),
    "code": reproduction_code,
    "raw_output": reproduction_raw,
    "rag_decision": (
        reproduction_outcome.decision.reason
    ),
    "sources": reproduction_outcome.sources,
    "verification_note": (
        "Python numeric separators are normalized before "
        "comparison; 250000 and 250_000 are equivalent."
    ),
}

print("Grounded reproduction:", reproduction_ok)
print()
print(reproduction_code)


# ============================================================
# Test 2 — Wrapper/composition correction
# ============================================================

print()
print("=" * 70)
print("TEST 2 — WRAPPER COMPOSITION")
print("=" * 70)

composition_query = (
    "Implement flag_for_manual_review("
    "amount, customer_tenure_days, "
    "destination_country, trusted_device, "
    "threshold=70"
    "). The new function must call this repository's "
    "transfer_risk_score function and return True when "
    "the resulting score is greater than or equal to "
    "threshold, otherwise False. Do not reproduce or copy "
    "the dependency's internal implementation. "
    "Return Python code only."
)

(
    composition_code,
    composition_checked,
    composition_outcome,
    composition_raw,
) = generate_with_rag(composition_query)

composition_ok = (
    bool(composition_checked.get("valid"))
    and "def flag_for_manual_review" in composition_code
    and "transfer_risk_score(" in composition_code
    and "threshold" in composition_code
    and (
        ">= threshold" in composition_code
        or "threshold <=" in composition_code
    )
    and "high_risk_countries" not in composition_code
    and "250_000" not in composition_code
    and "250000" not in composition_code
)

verification_results["wrapper_composition"] = {
    "success": composition_ok,
    "status": composition_checked.get("status"),
    "reason": composition_checked.get("reason"),
    "code": composition_code,
    "raw_output": composition_raw,
    "rag_decision": (
        composition_outcome.decision.reason
    ),
    "sources": composition_outcome.sources,
    "verification_note": (
        "The new wrapper must call the retrieved dependency "
        "rather than copying its implementation."
    ),
}

print("Wrapper composition:", composition_ok)
print()
print(composition_code)


# ============================================================
# Test 3 — In-place extension correction
# ============================================================

print()
print("=" * 70)
print("TEST 3 — IN-PLACE POLICY EXTENSION")
print("=" * 70)

extension_query = (
    "Extend this repository's transfer_risk_score function "
    "in place. Preserve every existing threshold, weight, "
    "country rule and cap. Add the boolean parameter "
    "is_weekend. When is_weekend is true, add 10 to the "
    "calculated risk score before applying the existing "
    "100-point cap. Return the complete updated Python "
    "function only."
)

(
    extension_code,
    extension_checked,
    extension_outcome,
    extension_raw,
) = generate_with_rag(extension_query)

extension_numeric_source = (
    normalized_numeric_source(extension_code)
)

has_weekend_rule = (
    "is_weekend" in extension_code
    and (
        "+= 10" in extension_code
        or "+ 10" in extension_code
        or "10 +" in extension_code
    )
)

has_final_cap = (
    "min(" in extension_code
    and "100" in extension_numeric_source
)

extension_ok = (
    bool(extension_checked.get("valid"))
    and contains_original_transfer_policy(
        extension_code
    )
    and has_weekend_rule
    and has_final_cap
)

verification_results["inplace_extension"] = {
    "success": extension_ok,
    "status": extension_checked.get("status"),
    "reason": extension_checked.get("reason"),
    "code": extension_code,
    "raw_output": extension_raw,
    "rag_decision": (
        extension_outcome.decision.reason
    ),
    "sources": extension_outcome.sources,
    "verification_note": (
        "The updated implementation must preserve the "
        "original policy and add the weekend rule before "
        "the existing cap."
    ),
}

print("In-place extension:", extension_ok)
print()
print(extension_code)


# ============================================================
# Final verification outcome
# ============================================================

overall_success = (
    reproduction_ok
    and composition_ok
    and extension_ok
)

print()
print("=" * 70)
print("V1.3 TRANSFORMATION VERIFICATION")
print("=" * 70)
print("Grounded reproduction :", reproduction_ok)
print("Wrapper composition   :", composition_ok)
print("In-place extension    :", extension_ok)
print("Overall success       :", overall_success)
print("=" * 70)

assert reproduction_ok, (
    "v1.3 genuinely regressed on grounded reproduction."
)

assert composition_ok, (
    "v1.3 still copied evidence instead of composing "
    "the requested wrapper."
)

assert extension_ok, (
    "v1.3 still copied evidence instead of applying "
    "the requested in-place extension."
)

print("V1.3 TRANSFORMATION CORRECTION: PASS")

Loading baseline model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:100: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Grounded reproduction: True

def transfer_risk_score(amount, customer_tenure_days, destination_country, trusted_device):
    high_risk_countries = {"IR", "KP", "SY"}
    score = 0.0
    if amount >= 250000:
        score += 40
    elif amount >= 100000:
        score += 25
    if customer_tenure_days < 30:
        score += 20
    if destination_country.strip().upper() in high_risk_countries:
        score += 30
    if not trusted_device:
        score += 15
    return min(score, 100.0)

TEST 2 — WRAPPER COMPOSITION
Wrapper composition: True

def flag_for_manual_review(amount, customer_tenure_days, destination_country, trusted_device, threshold=70):
    score = transfer_risk_score(amount, customer_tenure_days, destination_country, trusted_device)
    return score >= threshold

TEST 3 — IN-PLACE POLICY EXTENSION
In-place extension: True

def transfer_risk_score(amount, customer_tenure_days, destination_country, trusted_device, is_weekend):
    """Calculate the repository's 0-100 cross-bo

## Step 8 — Save the verification report

The report records generated code and retrieval provenance for mentor review.


In [14]:
# ============================================================
# Step 8 — Save verification report
# ============================================================

import json
import time

report = {
    "adapter_name": V13_ADAPTER_NAME,
    "parent_adapter_name": V12_ADAPTER_NAME,
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "training_strategy": "v1.2 continuation with balanced replay, composition and extension",
    "verification_results": verification_results,
    "overall_success": overall_success,
}
report_path = storage.path("outputs/reports/rag_transform_retrain_v1_3_verification.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved:", report_path)


Saved: /content/drive/MyDrive/RepoCoderStudio/outputs/reports/rag_transform_retrain_v1_3_verification.json


## Step 9 — Launch the enhanced Gradio interface

This uses the v1.3 runtime already loaded above. Python outputs can be executed, Java outputs can be compiled/run, and Java code is displayed with Java formatting. These browser actions are demonstration checks; Docker remains the stronger isolated evaluator.


In [ ]:
# ============================================================
# Step 9 — Launch Gradio with v1.3 and browser execution
# ============================================================

import importlib
import gradio as gr
import src.browser_code_runner as runner_module
import src.gradio_runtime as runtime_module
import src.gradio_showcase as showcase_module

runner_module = importlib.reload(runner_module)
showcase_module = importlib.reload(showcase_module)
runtime_module.patch_gradio_template_compatibility()

for _name in ("demo", "gradio_demo"):
    _previous = globals().get(_name)
    if _previous is not None and hasattr(_previous, "close"):
        try:
            _previous.close()
        except Exception:
            pass
try:
    gr.close_all()
except Exception:
    pass

ui_kwargs = {
    "gen_engine": engine,
    "baseline_model": baseline_model,
    "baseline_tokenizer": baseline_tokenizer,
    "finetuned_model": finetuned_model,
    "finetuned_tokenizer": finetuned_tokenizer,
    "retrieval_engines": {"ledgerflow": ledgerflow_retrieval},
    "task_registry": task_registry,
    "output_validator": output_validator,
    "project_root": PROJECT_ROOT,
}
demo = showcase_module.build_gradio_showcase(**ui_kwargs)
demo.queue(default_concurrency_limit=1, max_size=20)

print("Launching RepoCoderStudio with the v1.3 transform-aware adapter...")
print("Python can run and Java can compile/run from each result card.")
print("Open the fresh gradio.live link below in a new browser tab.")
demo.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True,
    prevent_thread_lock=True,
)


Launching RepoCoderStudio with the v1.3 transform-aware adapter...
Python can run and Java can compile/run from each result card.
Open the fresh gradio.live link below in a new browser tab.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://4c211cb48973f24d4f.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
07:42:28 | WARNING  | RepoCoderStudio.Main | Code generation ended before an implementation was produced; retrying once with a guarded minimum completion length.
07:43:09 | WARNING  | RepoCoderStudio.Main | Code generation ended before an implementation was produced; retrying once with a guarded minimum completion length.
